In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/post_cell_9.pickle

In [ ]:
%%cudf.pandas.profile
### cell 10 ###

# 1. Top 11 countries by total titles (use groupby.count instead of value_counts)
country_order = (
    df_cell6
    .groupby('first_country')['type']
    .count()
    .sort_values(ascending=False)
    .head(11)
    .index
)

# 2. Build boolean indicator columns and sum per country (all on GPU)
counts = (
    df_cell6
    .assign(
        Movie    = df_cell6['type'] == 'Movie',
        TV_Show  = df_cell6['type'] == 'TV Show'
    )
    .groupby('first_country')[['Movie', 'TV_Show']]
    .sum()
)

# 3. Restrict to top 11 and rename column
data_q2q3 = (
    counts
    .loc[country_order]
    .rename(columns={'TV_Show': 'TV Show'})
)

# 4. Compute total and ratio columns elementwise on GPU
data_q2q3['sum'] = data_q2q3['Movie'] + data_q2q3['TV Show']

data_q2q3_ratio = data_q2q3[['Movie', 'TV Show']].copy()
data_q2q3_ratio['Movie']   = data_q2q3_ratio['Movie']   / data_q2q3['sum']
data_q2q3_ratio['TV Show'] = data_q2q3_ratio['TV Show'] / data_q2q3['sum']

# 5. Sort by movie ratio
data_q2q3_ratio = data_q2q3_ratio.sort_values('Movie', ascending=True)